# 04 - Chlorophyll-a Data Preparation (MODIS AQUA)

Processes daily MODIS AQUA chlorophyll-a satellite files into per-year netCDF files with a time dimension.

**Steps:**
1. Scan the source folder for all daily AQUA MODIS Chl-a netCDF files
2. Group files by year
3. Concatenate daily observations into a single dataset per year
4. Save yearly files: `chl_daily_{year}.nc`

In [1]:
import xarray as xr
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from pyproj import datadir
datadir.set_data_dir("/home/jupyter-daniela/.conda/envs/peru_environment/share/proj")

src_path = Path("/home/jupyter-daniela/suyana/sources/chl_peru")
out_path = Path("/home/jupyter-daniela/suyana/peru_production/features")
out_path.mkdir(parents=True, exist_ok=True)

In [2]:
all_files = sorted(src_path.glob("AQUA_MODIS.*.nc"))
print(f"Total files: {len(all_files)}")

files_by_year = {}
for f in all_files:
    year = f.name.split('.')[1][:4]
    files_by_year.setdefault(year, []).append(f)

print(f"Years found: {sorted(files_by_year.keys())}")

Total files: 8583
Years found: ['2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']


In [ ]:
for year, files in tqdm(sorted(files_by_year.items()), desc="Years"):
    daily_arrays = []

    for f in sorted(files):
        date_str = f.name.split('.')[1]
        date = pd.Timestamp(date_str)
        try:
            ds = xr.open_dataset(f)
            chl = ds['chlor_a'].expand_dims(time=[date])
            daily_arrays.append(chl)
            ds.close()
        except Exception as e:
            print(f"  Skipping {f.name}: {e}")
            continue

    if not daily_arrays:
        print(f"{year}: no valid files, skipping")
        continue

    ds_year = xr.concat(daily_arrays, dim='time').to_dataset(name='chlor_a')
    ds_year = ds_year.sortby('time')
    out_file = out_path / f"chl_daily_{year}.nc"
    ds_year.to_netcdf(out_file)
    print(f"{year}: {ds_year.sizes['time']} days saved → {out_file.name}")

In [ ]:
# Compute chl climatology across all available years
YEARS = [int(y) for y in files_by_year.keys() if (out_path / f"chl_daily_{y}.nc").exists()]
print(f"Computing climatology from years: {sorted(YEARS)}")

datasets = [xr.open_dataset(out_path / f"chl_daily_{y}.nc") for y in sorted(YEARS)]
ds_all = xr.concat(datasets, dim='time').sortby('time')
ds_clim = ds_all.groupby('time.dayofyear').mean(dim='time').rename({'chlor_a': 'chl_clim'})
ds_clim.to_netcdf(out_path / 'chl_climatology_doy.nc')
print("Climatology saved -> chl_climatology_doy.nc")

In [ ]:
# Compute daily chl anomalies per year
ds_clim = xr.open_dataset(out_path / 'chl_climatology_doy.nc')

for year in tqdm(sorted(YEARS), desc="Anomalies"):
    anom_file = out_path / f"chl_anomaly_daily_{year}.nc"
    if anom_file.exists():
        print(f"{year}: already exists, skipping")
        continue
    ds_year = xr.open_dataset(out_path / f"chl_daily_{year}.nc")
    doy = ds_year['time'].dt.dayofyear
    clim_aligned = ds_clim['chl_clim'].sel(dayofyear=doy).drop_vars('dayofyear')
    anom = (ds_year['chlor_a'] - clim_aligned).to_dataset(name='chl_anomaly')
    anom.to_netcdf(anom_file)
    ds_year.close()
    print(f"{year}: anomaly saved → {anom_file.name}")

ds_clim.close()